In [ ]:
from pycwb.modules.report.read_results import read_triggers, read_live_time
from pycwb.modules.report.far_rho import far_rho
from pycwb.modules.report.report import report_zero_lag
from pycwb.modules.report.continues_poisson import get_percentiles, get_percentiles_ROOT
import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")


How to build a background for gravitational waves? Time slides!
![lags](./img/lags.png)

## Read the background ranking statistics and livetime

Due to the memory limitation on mybinder, the background ranking statistics and livetime are precomputed and stored in files. In a real analysis, you can compute them directly from the background triggers and livetime as shown in the commented code below. You can try it on your local machine with sufficient memory (>6GB).


In [11]:
# run_dir = 'background'

# background_triggers = read_triggers(work_dir='.',
#                                     run_dir=run_dir,
#                                     filters=["slag[0] != 0 or slag[1] != 0 or lag[0] != 0"])

# livetime = read_live_time(work_dir='.',
#                           run_dir=run_dir,
#                           filters=["shift[0] != 0 or shift[1] != 0 or lag != 0"])

# bkg_ranking_statistics = [t['rho'][0] for t in background_triggers if t['rho'][0] is not None]

Reading results from ./background/catalog/catalog.json
Removed 2509 duplicated events
number of triggers before filtering: 268295
Performing filter: slag[0] != 0 or slag[1] != 0 or lag[0] != 0
number of triggers after filtering: 268122
Reading live time from ./background/catalog/catalog.json
Removed 2509 duplicated events
number of live times before filtering: 83556
Performing filter: shift[0] != 0 or shift[1] != 0 or lag != 0
number of live times after filtering: 83496
Total live time: 100195200.0s (1159.67 days, 3.18 years)


In [12]:
import pickle

with open("background/intermediate_data/bkg_ranking_statistics.pkl", "rb") as f:
    bkg_ranking_statistics = pickle.load(f)

with open("background/intermediate_data/livetime.pkl", "rb") as f:
    livetime = pickle.load(f)


In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

bin_size = 0.01

hist, bins = np.histogram(bkg_ranking_statistics, bins=np.arange(min(bkg_ranking_statistics), max(bkg_ranking_statistics) + bin_size, bin_size))
accumulate_hist = np.cumsum(hist[::-1])[::-1]
bins = bins[:-1]

livetime_in_years = livetime / 86400 / 365.25
far = [accumulate_hist[i] / livetime_in_years for i in range(len(accumulate_hist))]

# plot the number of events vs ranking parameter
plt.plot(bins, hist, drawstyle='steps-post')
plt.xlabel('rho[0]')
plt.ylabel('number of events')
plt.yscale('log')
plt.show()
plt.close()

Accumulative plot

In [ ]:
plt.plot(bins, far, drawstyle='steps-post')
plt.xlabel('rho[0]')
plt.ylabel('far (1/yr)')
plt.yscale('log')
plt.show()
plt.close()

Now let's have a look at the zero-lag (non-shifted) triggers detected in the run.

In [ ]:
# zerolag_triggers = read_triggers(work_dir='.',
#                                     run_dir=run_dir,
#                                     filters=["lag[0]==0", "slag[0]==0", "slag[1]==0"])

Reading results from ./background_reduced/catalog/catalog.json
Removed 1777 duplicated events
number of triggers before filtering: 163855
Performing filter: lag[0]==0 and slag[0]==0 and slag[1]==0
number of triggers after filtering: 173


In [ ]:
# zero_lag_livetime = read_live_time(work_dir='.',
#                                   run_dir=run_dir,
#                                   filters=["shift[0]==0", "shift[1]==0", "lag==0"])

Reading live time from ./background/catalog/catalog.json
Removed 2509 duplicated events
number of live times before filtering: 83556
Performing filter: shift[0]==0 and shift[1]==0 and lag==0
number of live times after filtering: 60
Total live time: 72000.0s (0.83 days, 0.00 years)


In [18]:
with open('background/intermediate_data/zerolag_triggers.pkl', 'rb') as f:
    zerolag_triggers = pickle.load(f)

with open('background/intermediate_data/zero_lag_livetime.pkl', 'rb') as f:
    zero_lag_livetime = pickle.load(f)

In [ ]:
ranking_statistics = [t['rho'][0] for t in zerolag_triggers if t['rho'][0] is not None]

# align the values to the bin
bin_indices = np.digitize(ranking_statistics, bins) - 1
far_values = [far[i] for i in bin_indices]
for i, trigger in enumerate(zerolag_triggers):
    trigger['far'] = far_values[i]

print(f"Plotting far vs rho")
plt.scatter(ranking_statistics, far_values)
plt.xlabel('rho')
plt.ylabel('far')
plt.yscale('log')
plt.show()
plt.close()

In [ ]:
ranked_triggers = sorted(zerolag_triggers, key=lambda x: 1/x['far'], reverse=True)
ifar = np.array([1/trigger['far'] for trigger in ranked_triggers])
plot_y = np.arange(1, len(ranked_triggers) + 1)
plt.plot(ifar, plot_y, drawstyle='steps-post')

livetime_in_years = zero_lag_livetime / 86400 / 365.25
ifar_min = min(ifar)
n_events_at_ifar_min = livetime_in_years / ifar_min
ifar_max = max(ifar)
n_events_at_ifar_max = livetime_in_years / ifar_max
plt.plot([ifar_min, ifar_max], [n_events_at_ifar_min, n_events_at_ifar_max], color='black', linewidth=0.5)


ifar_range = np.linspace(ifar_min, ifar_max, 500)
n_events_range = livetime_in_years / ifar_range
sigma_levels = [1, 2, 3]
confidence_levels = [0.6827, 0.9545, 0.9973]
precentiles = np.array([[(1 - c) / 2, 1 - (1 - c) / 2] for c in confidence_levels]).reshape(-1)
#conf_intervals = {sigma: poisson.interval(confidence, n_events_range) for sigma, confidence in zip(sigma_levels, confidence_levels)}
conf_intervals = np.array([get_percentiles_ROOT(n, precentiles).reshape((len(confidence_levels), 2)) for n in n_events_range])
# Plot the Poisson confidence intervals
colors = ['gray', 'gray', 'gray']
for sigma, color in zip(sigma_levels, colors):
    lower, upper = conf_intervals[:, sigma-1, 0], conf_intervals[:, sigma-1, 1]
    plt.fill_between(ifar_range, lower, upper, color=color, alpha=0.6 / sigma, label=f'{sigma} sigma', linewidth=0.4,interpolate=True)

plt.xlim(ifar_min, ifar_max)
plt.ylim(6e-1, max(n_events_range))
plt.xscale('log')
plt.yscale('log')

In [ ]:
from gwpy.time import from_gps
from pprint import pprint

zerolag_triggers_sorted = sorted(zerolag_triggers, key=lambda x: x['rho'][0], reverse=True)
print(from_gps(zerolag_triggers_sorted[0]['gps'][0]).isoformat())

In [ ]:
pprint(zerolag_triggers_sorted[0])